<a href="https://colab.research.google.com/github/mebre7/house-price-predictor-mlops/blob/colab_branch/notebooks/02_feature_engineering_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering and Model Training
Feature engineering is the process of using domain knowledge to create new features from existing data. It involves transforming raw data into a format that is more suitable for machine learning algorithms, which can lead to improved model performance.
- In this notebook of the house price predictor project, I will be performing feature engineering on the dataset to create new features that can help improve the predictive power of the model. This may include creating interaction terms, polynomial features, or aggregating existing features to capture more complex relationships in the data.

## 1. Import Packages & Environment Setup

In [ ]:
%pip install catboost
%pip install xgboost
%pip install lightgbm

In [ ]:
import os
import sys
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from google.colab import userdata

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 2. Data Ingestion

In [ ]:
# 1. Load credentials from google colab credentials

DB_USER = userdata.get("SUPABASE_USER")
DB_PASSWORD = quote_plus(userdata.get("SUPABASE_PASSWORD"))
DB_HOST = userdata.get("SUPABASE_HOST")
DB_PORT = userdata.get("SUPABASE_PORT")
DB_NAME = userdata.get("SUPABASE_NAME")

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL)
    with engine.connect() as conn:
        df = pd.read_sql(text("SELECT * FROM housing_data"), conn)
    print("Data loaded successfully from Supabase PostgreSQL database.")
except Exception as e:
    print(f"PostgreSQL connection failed ({e}). Loading from raw CSV fallback...")
    raw_path = os.path.join("..", "data", "raw", "AmesHousing.csv")
    if not os.path.exists(raw_path):
        raw_path = os.path.join("data", "raw", "AmesHousing.csv")
    df = pd.read_csv(raw_path)
    print(f"Data loaded successfully from local CSV: {raw_path}")

# Drop PID / Order columns
drop_cols = [c for c in ["Order", "PID", "Id"] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
df.info()

## 3. Features classification

**1. Numerical features**

In [ ]:
# define numerical features
numerical_features_all = df.select_dtypes(include=[np.number]).columns.tolist()

In [ ]:
len(numerical_features_all)

In [ ]:
len(df.select_dtypes(include=["float64", "int64"]).columns.to_list())

In [ ]:
# Discrete numerical features
discrete_num_features = [
    'Bsmt_Full_Bath',
    'Bsmt_Half_Bath',
    'Full_Bath',
    'Half_Bath',
    'Bedroom_AbvGr',
    'Kitchen_AbvGr',
    'TotRms_AbvGrd',
    'Fireplaces',
    'Garage_Cars'
]

In [ ]:
# Continuous numerical features
continuous_num_features = [
    'Lot_Frontage',
    'Lot_Area',
    'Mas_Vnr_Area',
    'BsmtFin_SF_1',
    'BsmtFin_SF_2',
    'Bsmt_Unf_SF',
    'Total_Bsmt_SF',
    '1st_Flr_SF',
    '2nd_Flr_SF',
    'Low_Qual_Fin_SF',
    'Gr_Liv_Area',
    'Garage_Area',
    'Wood_Deck_SF',
    'Open_Porch_SF',
    'Enclosed_Porch',
    '3Ssn_Porch',
    'Screen_Porch',
    'Pool_Area',
    'Misc_Val'
]

In [ ]:
num_features_only = discrete_num_features + continuous_num_features
print(f'{len(num_features_only)} numerical features')

**2. Categorical features**

In [ ]:
# Nominal categorical features
nominal_cat_features = [
    'MS_SubClass',
    'MS_Zoning',
    'Street',
    'Alley',
    'Land_Contour',
    'Utilities',
    'Lot_Config',
    'Neighborhood',
    'Condition_1',
    'Condition_2',
    'Bldg_Type',
    'House_Style',
    'Roof_Style',
    'Roof_Matl',
    'Exterior_1st',
    'Exterior_2nd',
    'Mas_Vnr_Type',
    'Foundation',
    'Heating',
    'Central_Air',
    'Electrical',
    'Garage_Type',
    'Paved_Drive',
    'Fence',
    'Misc_Feature',
    'Sale_Type',
    'Sale_Condition'
]

In [ ]:
# Ordinal categorical features
ordinal_cat_features = [
    'Lot_Shape',
    'Land_Slope',
    'Overall_Qual',
    'Overall_Cond',
    'Exter_Qual',
    'Exter_Cond',
    'Bsmt_Qual',
    'Bsmt_Cond',
    'Bsmt_Exposure',
    'BsmtFin_Type_1',
    'BsmtFin_Type_2',
    'Heating_QC',
    'Kitchen_Qual',
    'Functional',
    'Fireplace_Qu',
    'Garage_Finish',
    'Garage_Qual',
    'Garage_Cond',
    'Pool_QC'
]

In [ ]:
categorical_features = nominal_cat_features + ordinal_cat_features
print(f'{len(categorical_features)} categorical featurese')

In [ ]:
set(categorical_features).intersection(set(discrete_num_features))

**Change data types of `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [ ]:
for col in categorical_features:
    df[col] = df[col].astype('category')

In [ ]:
df[categorical_features].info()

**3. Temporal features**

In [ ]:
date_year = ['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', ]
date_month = ['Mo_Sold']

for col in date_year:
    df[col] = pd.to_datetime(df[col], format='%Y').dt.year

for col in date_month:
    df[col] = pd.to_datetime(df[col], format='%m').dt.month

In [ ]:
# Temporal features
temporal_features = date_year + date_month
temporal_features

**4. Target feature**

In [ ]:
target = ['SalePrice']

**All features**

In [ ]:
all_features = discrete_num_features + continuous_num_features + nominal_cat_features + ordinal_cat_features + temporal_features + target

In [ ]:
len(all_features)

In [ ]:
len(df.columns.to_list())

**Summarize**

In [ ]:
print(len(discrete_num_features))
print(len(continuous_num_features))
print(len(nominal_cat_features))
print(len(ordinal_cat_features))
print(len(temporal_features))
print(len(target))

**Total features = 80**
- **Total numerical features = $37$ ($11$ discrete, $19$ continuous, $5$ temporal, $1$ target, $1$ categorical)**
- discrete numerical features = $9$
- continuous features = $37 - 11 - 5 - 1 - 1 = 19$
- temporal features = $5$
- target feature = $1$
- **total categorical features = $46$ ($2$ from discrete)**
- ordinal categorical features = $19$
- nominal categorial features = $27$

-> **Total features $= 9 + 19 + 5 + 1 + 19 + 27=80$**

In [ ]:

# print columns
print(f'{len(discrete_num_features)} discrete numerical features: {discrete_num_features}')
print(f'\n{len(continuous_num_features)} continuous numerical features: {continuous_num_features}')
print(f'\n{len(ordinal_cat_features)} ordinal categorical features: {ordinal_cat_features}')
print(f'\n{len(nominal_cat_features)} nominal categorical features: {nominal_cat_features}')
print(f'\n{len(temporal_features)} temporal features: {temporal_features}')
print(f'\n{len(target)} target feature: {target}')

In [ ]:
# total
print(f'Numer of all features: {len(discrete_num_features) + len(continuous_num_features) + len(ordinal_cat_features) + len(nominal_cat_features) + len(temporal_features) + len(target)}')

## 4. Train-Test split

In [ ]:
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
y_train.head()

In [ ]:
y_test.head()

## 5. Missing Value Handling (Imputation)

In [ ]:
count = X.isnull().sum().sort_values(ascending=False)
percentage = ((X.isnull().sum() / df.isnull().count()) * 100).sort_values(ascending=False)
null_values = pd.concat([count, percentage], axis=1, keys=['Count', 'Percentage'])

In [ ]:
null_values.head()

**Visualize missing values**

In [ ]:
fig = px.bar(x=null_values.index, y=null_values['Percentage'],
             labels={'x': 'Features', 'y': 'Percent of missing values'},
             title='Percent missing data by feature')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**1. Structural Absence (Replace `NaN` with `None` or `0`)**

These columns use missing values to denote that the feature does not exist on the property.

- **`structural_categorical` (Fill with `None`)**

In [ ]:
structural_categorical = [
    'Alley',

    'Bsmt_Qual',
    'Bsmt_Cond',
    'Bsmt_Exposure',
    'BsmtFin_Type_1',
    'BsmtFin_Type_2',

    'Fireplace_Qu',

    'Garage_Type',
    'Garage_Finish',
    'Garage_Qual',
    'Garage_Cond',

    'Pool_QC',
    'Fence',
    'Misc_Feature'
]

```
df.Alley=["No Alley" if x is np.nan else x for x in df.Alley]
df.Mas_Vnr_Type=["Missing" if x is np.nan else x for x in df.Mas_Vnr_Type]
df.Bsmt_Qual=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Qual]
df.Bsmt_Cond=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Cond]
df.Bsmt_Exposure=["No Bsmnt" if x is np.nan else x for x in df.Bsmt_Exposure]
df.BsmtFin_Type_1=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_1]
df.BsmtFin_Type_2=["No Bsmnt" if x is np.nan else x for x in df.BsmtFin_Type_2]
df.Fireplace_Qu=["No Firepl" if x is np.nan else x for x in df.Fireplace_Qu]
df.Garage_Type=["No Grg" if x is np.nan else x for x in df.Garage_Type]
df.Garage_Finish=["No Grg" if x is np.nan else x for x in df.Garage_Finish]
df.Garage_Qual=["No Grg" if x is np.nan else x for x in df.Garage_Qual]
df.Garage_Cond=["No Grg" if x is np.nan else x for x in df.Garage_Cond]
df.Pool_QC=["No Pool" if x is np.nan else x for x in df.Pool_QC]
df.Fence=["No Fence" if x is np.nan else x for x in df.Fence]
df.Misc_Feature=["None" if x is np.nan else x for x in df.Misc_Feature]
```

```py
df['Alley'] = df['Alley'].fillna('No Alley')
df['Mas_Vnr_Type'] = df['Mas_Vnr_Type'].fillna('Missing')
df['Bsmt_Qual'] = df['Bsmt_Qual'].fillna('No Bsmnt')
df['Bsmt_Cond'] = df['Bsmt_Cond'].fillna('No Bsmnt')
df['Bsmt_Exposure'] = df['Bsmt_Exposure'].fillna('No Bsmnt')
df['BsmtFin_Type_1'] = df['BsmtFin_Type_1'].fillna('No Bsmnt')
df['BsmtFin_Type_2'] = df['BsmtFin_Type_2'].fillna('No Bsmnt')
df['Fireplace_Qu'] = df['Fireplace_Qu'].fillna('No Firepl')
df['Garage_Type'] = df['Garage_Type'].fillna('No Grg')
df['Garage_Finish'] = df['Garage_Finish'].fillna('No Grg')
df['Garage_Qual'] = df['Garage_Qual'].fillna('No Grg')
df['Garage_Cond'] = df['Garage_Cond'].fillna('No Grg')
df['Pool_QC'] = df['Pool_QC'].fillna('No Pool')
df['Fence'] = df['Fence'].fillna('No Fence')
df['Misc_Feature'] = df['Misc_Feature'].fillna('None')
```

In [ ]:
fill_values = {
    'Alley': 'No Alley Access',
    'Bsmt_Qual': 'No Basement',
    'Bsmt_Cond': 'No Basement',
    'Bsmt_Exposure': 'No Basement',
    'BsmtFin_Type_1': 'No Basement',
    'BsmtFin_Type_2': 'No Basement',
    'Fireplace_Qu': 'No Fireplace',
    'Garage_Type': 'No Garage',
    'Garage_Finish': 'No Garage',
    'Garage_Qual': 'No Garage',
    'Garage_Cond': 'No Garage',
    'Pool_QC': 'No Pool',
    'Fence': 'No Fence',
    'Misc_Feature': 'None'
}
for col, fill_value in fill_values.items():
    if fill_value not in X[col].cat.categories:
        X[col] = X[col].cat.add_categories(fill_value)
    X[col] = X[col].fillna(fill_value)

In [ ]:
X[structural_categorical].isnull().sum()

- **`structural_numerical` (Fill with `0`)**

In [ ]:
structural_numerical = [
    'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF',
    'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Garage_Cars', 'Garage_Area', 'Mas_Vnr_Area'
]

In [ ]:
for col in structural_numerical:
  X[col] = X[col].fillna(0)

In [ ]:
X[structural_numerical].isnull().sum()

**Checking**

In [ ]:
X[structural_numerical + structural_categorical].isnull().sum().sum()

In [ ]:
len(structural_numerical + structural_categorical)

In [ ]:
X[structural_categorical + structural_numerical].head()

In [ ]:
features_with_missing_values = list(set(X.columns.to_list()) - set(structural_numerical + structural_categorical))
print(len(features_with_missing_values))
features_with_missing_values

> Out of $80$ total features, $22$ features with missing values are handled considering _structuring absence_. Remaining $58$ will be handled by _statistical imputation_ and other ways.

In [ ]:
X.head()

In [ ]:
# updated X_train and X_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**2. Statistical Imputation (True Missingness/Clerical Errors)**

These features are physically required for a house to exist. Missing data here represents a processing omission or clerical typo.

- **`spatial_numerical` (Impute using the median of their specific `Neighborhood`)**

In [ ]:
spatial_numerical = ['Lot_Frontage']

In [ ]:
X_train[spatial_numerical].isnull().sum()

In [ ]:
X_test[spatial_numerical].isnull().sum()

In [ ]:
X[spatial_numerical].isnull().sum()

In [ ]:
# Calculate the median Lot_Frontage per Neighborhood from X_train
neighborhood_medians = X_train.groupby('Neighborhood')['Lot_Frontage'].median()

In [ ]:
# fill missing values in X_train using the training medians
X_train['Lot_Frontage'] = X_train['Lot_Frontage'].fillna(X_train['Neighborhood']).map(neighborhood_medians)

In [ ]:
# Fill missing values in X_test using the *exact same* training medians
X_test['Lot_Frontage'] = X_test['Lot_Frontage'].fillna(X_test['Neighborhood'].map(neighborhood_medians))

In [ ]:
# Fallback: If a neighborhood in X_test happens to have all NaNs or a new category
# not seen in training, fill remaining NaNs with the overall X_train median
global_median = X_train['Lot_Frontage'].median()
X_train['Lot_Frontage'] = X_train['Lot_Frontage'].fillna(global_median)
X_test['Lot_Frontage'] = X_test['Lot_Frontage'].fillna(global_median)

In [ ]:
X_train[spatial_numerical].isnull().sum()

In [ ]:
X_test[spatial_numerical].isnull().sum()

In [ ]:
# update whole dataset
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
X.head()

- **`mode_categorical` (Impute using the column's global mode/most frequent item)**

In [ ]:
mode_categorical = [
    'Utilities', 'Exterior_1st', 'Exterior_2nd', 'Electrical',
    'Kitchen_Qual', 'Functional', 'Sale_Type'
]

In [ ]:
X[mode_categorical].isnull().sum()

In [ ]:
X[mode_categorical].head()

In [ ]:
for col in mode_categorical:
    if df[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])

- **`special_case_temporal` (Impute missing indices dynamically with that row's `Year_Built` value)**

In [ ]:
special_case_temporal = ['Garage_Yr_Blt']

In [ ]:
X[temporal_features].isnull().sum()

In [ ]:
X[temporal_features].head()

In [ ]:
X['Garage_Yr_Blt'] = X['Garage_Yr_Blt'].fillna(X['Year_Built'])

In [ ]:
X[temporal_features].isnull().sum()

The data dictionary describes `MasVnrType` as masonry veneer type and `MasVnrArea` as veneer area.

In [ ]:
X[['Mas_Vnr_Area', 'Mas_Vnr_Area']].isnull().sum()

In [ ]:
X.loc[
    X["Mas_Vnr_Type"].eq("None") &
    X["Mas_Vnr_Area"].isna(),
    "Mas_Vnr_Area"
] = 0

# Correcting the imputation for Mas_Vnr_Type
# It was previously handled in the code block PIIK1l3yFmJ6 (markdown)
# We ensure 'Missing' is a category and fill NaNs with it.
if 'Missing' not in X['Mas_Vnr_Type'].cat.categories:
    X['Mas_Vnr_Type'] = X['Mas_Vnr_Type'].cat.add_categories('Missing')
X['Mas_Vnr_Type'] = X['Mas_Vnr_Type'].fillna('Missing')


**Final check for existence of null values**

In [ ]:
# Null values left?
print(X.isnull().values.any())
if X.isnull().values.any():
  print("Yes, there exist null values!")
  null_counts = X.isnull().sum()
  null_counts = null_counts[null_counts > 0]
  print(null_counts)
else:
  print("No null values")

> **Final dataset with no missing values**

In [ ]:
# updated X_train and X_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X.head()

In [ ]:
X.info()

## 6. Outlier Removal/Resolving

Important concepts in outlier detection! Here's a breakdown:

*   **Interquartile Range (IQR):** The IQR is a measure of statistical dispersion, representing the range of the middle 50% of your data. It's calculated as the difference between the third quartile (Q3) and the first quartile (Q1).

    *   **Q1 (First Quartile):** The value below which 25% of the data falls.
    *   **Q3 (Third Quartile):** The value below which 75% of the data falls.
    *   `IQR = Q3 - Q1`

*   **Outlier Thresholds (Lower Bound and Upper Bound):** These are calculated boundaries used to identify potential outliers in a dataset. They are derived using the IQR.

    *   **Lower Bound:** `Q1 - 1.5 * IQR`
    *   **Upper Bound:** `Q3 + 1.5 * IQR`

*   **Outlier vs. IQR:**
    *   The **IQR** itself is a measure of the spread of the central portion of the data, robust to extreme values.
    *   **Outliers** are data points that fall outside the calculated lower and upper bounds. The 1.5 multiplier in the threshold calculation is a commonly used convention (often referred to as the "1.5 IQR rule") to define what constitutes an outlier relative to the bulk of the data's spread. Any data point less than the lower bound or greater than the upper bound is considered an outlier.

In [ ]:
num_features_only

In [ ]:
outlier_features = continuous_num_features

In [ ]:
def outlier_thresholds(X, column):
    Q1 = X[column].quantile(0.25)
    Q3 = X[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

In [ ]:
# detect outliers
def detect_outliers(X, column, low, up):
    return (X[column] < low) | (X[column] > up)

In [ ]:
def inspect_outliers(X, col_list, thresholds):
    results = {}
    for col in col_list:
        low, up = thresholds[col]
        results[col] = detect_outliers(X, col, low, up).any()
    return results

In [ ]:
def display_outlier_info(X, col_list, thresholds):
    print("Outlier thresholds for the numeric column")
    for col in col_list:
        low, up = thresholds[col]
        print(f"{col} : Lower Bound: {low}, Upper Bound: {up}")

    outliers_absent = []
    outliers_present = []

    for col, value in inspect_outliers(X, col_list, thresholds).items():
        if value:
            outliers_present.append(col)
        else:
            outliers_absent.append(col)

    print("\nColumns with outliers")
    print(outliers_present)
    print(f"column count: {len(outliers_present)}")

    print("\nColumns with no outliers")
    print(outliers_absent)
    print(f"column count: {len(outliers_absent)}")

    return outliers_present, outliers_absent

In [ ]:
len(numerical_features_all)

In [ ]:
len(discrete_num_features)

In [ ]:
len(num_features_only)

> $26 = 37 - 12 + 1$

In [ ]:
def threshold_resolution(X, column, low, up):
    X.loc[X[column] < low, column] = low
    X.loc[X[column] > up, column] = up

In [ ]:
thresholds = {}

for col in outlier_features:
    thresholds[col] = outlier_thresholds(X_train, col)

In [ ]:
outliers_df = pd.DataFrame(index=X_train.index)

for col, (low, up) in thresholds.items():
    outliers_df[col] = detect_outliers(X_train, col, low, up)

print("Number of outliers per numerical column:")
print(outliers_df.sum())

In [ ]:
# inside X_train
outliers_present, outliers_absent = display_outlier_info(
    X_train,
    outlier_features,
    thresholds
)

In [ ]:
# in X_test
outliers_present, outliers_absent = display_outlier_info(X_test, outlier_features, thresholds)

**Now remove outliers**

In [ ]:
for col, (low, up) in thresholds.items():
    threshold_resolution(X_train, col, low, up)
    threshold_resolution(X_test, col, low, up)

In [ ]:
# in X_train
outliers_present, outliers_absent = display_outlier_info(X_train, outlier_features, thresholds)

In [ ]:
# in X_test
outliers_present, outliers_absent = display_outlier_info(X_test, outlier_features, thresholds)

> **Now no columns with outliers except `SalePrice`**

In [ ]:
outliers_present, outliers_absent = display_outlier_info(X, outlier_features, thresholds)

## Update X or X_train and X_test
1. Update `X`
```py
# updated X
X = pd.concat([X_train, X_test], axis=0)
```
2. Update `X_train` and `X_test`
```py
# updated X_train and X_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

In [ ]:
# updated X
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
outliers_present, outliers_absent = display_outlier_info(X_test, outlier_features, thresholds)

## 7. Feature Creation / Engineering

In [ ]:
import numpy as np

def create_features(df):
    # Total square footage
    df['TotalSF'] = (
        df['1st_Flr_SF'] +
        df['2nd_Flr_SF'] +
        df['Total_Bsmt_SF']
    )

    # Age of house at time of sale
    df['Age'] = df['Yr_Sold'] - df['Year_Built']

    # Total number of bathrooms
    df['TotalBathrooms'] = (
        df['Full_Bath'].astype(float) +
        0.5 * df['Half_Bath'].astype(float) +
        df['Bsmt_Full_Bath'].astype(float) +
        0.5 * df['Bsmt_Half_Bath'].astype(float)
    )

    # Pool presence
    df['HasPool'] = (df['Pool_Area'] > 0).astype('category')

    # Total porch area
    df['TotalPorchSF'] = (
        df['Open_Porch_SF'] +
        df['Enclosed_Porch'] +
        df['3Ssn_Porch'] +
        df['Screen_Porch']
    )

    # Total parking capacity
    df['TotalParking'] = (
        df['Garage_Cars'].astype(float) +
        (df['Paved_Drive'] == 'Y').astype(int)
    )

    # Remodeling indicator
    df['Remodeled'] = (
        df['Year_Remod/Add'] != df['Year_Built']
    ).astype('category')

    # Years since last remodel
    df['YearsSinceRemodel'] = (
        df['Yr_Sold'] - df['Year_Remod/Add'].astype(float)
    )

    # Total rooms
    df['TotalRooms'] = df['TotRms_AbvGrd'].astype(float)

    # Living-area / lot-area ratio
    df['LotRatio'] = df['Gr_Liv_Area'] / df['Lot_Area']

    # Categorical age group
    def age_group(age):
        if age <= 10:
            return 'New'
        elif age <= 30:
            return 'Modern'
        elif age <= 50:
            return 'Mature'
        else:
            return 'Old'

    df['AgeGroup'] = df['Age'].apply(age_group).astype('category')

    # Total outdoor living space
    df['TotalOutdoorSF'] = (
        df['Wood_Deck_SF'] +
        df['TotalPorchSF']
    )

    # Fireplace presence
    df['HasFireplace'] = (
        df['Fireplaces'] > 0
    ).astype('category')

    # Overall quality category
    overall_qual_map = {
        10: 'Ex',
        9: 'Ex',
        8: 'Gd',
        7: 'Gd',
        6: 'TA',
        5: 'TA',
        4: 'Fa',
        3: 'Fa',
        2: 'Po',
        1: 'Po'
    }

    df['OverallQualCategory'] = (
        df['Overall_Qual']
        .map(overall_qual_map)
    )

    return df

In [ ]:
X_train = create_features(X_train)
X_test = create_features(X_test)

In [ ]:
quality_map = {
    'Ex': 5,
    'Gd': 4,
    'TA': 3,
    'Fa': 2,
    'Po': 1
}

for data in [X_train, X_test]:
    data['Exter_Qual'] = data['Exter_Qual'].map(quality_map)
    data['Kitchen_Qual'] = data['Kitchen_Qual'].map(quality_map)

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
X[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual','Kitchen_Qual']].dtypes

In [ ]:
X_train['Quality_Score'] = (
    X_train['Overall_Qual'].astype(int) +
    X_train['Exter_Qual'].astype(int) +
    X_train['Kitchen_Qual'].astype(int)
) / 3

X_test['Quality_Score'] = (
    X_test['Overall_Qual'].astype(int) +
    X_test['Exter_Qual'].astype(int) +
    X_test['Kitchen_Qual'].astype(int)
) / 3

In [ ]:
# X_train['OverallQualCategory'] = X_train['OverallQualCategory'].astype('category')
# X_test['OverallQualCategory'] = X_test['OverallQualCategory'].astype('category')

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
X[['Overall_Qual', 'Quality_Score', 'OverallQualCategory', 'Exter_Qual','Kitchen_Qual']].dtypes

In [ ]:
X[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual','Kitchen_Qual']].tail()

In [ ]:
new_numerical_features = [
    'TotalSF',
    'Age',
    'TotalBathrooms',
    'TotalPorchSF',
    'YearsSinceRemodel',
    'TotalParking',
    'TotalRooms',
    'LotRatio',
    'TotalOutdoorSF'
]

In [ ]:
new_categorical_features = [
    'AgeGroup',
    'HasPool',
    'Remodeled',
    'HasFireplace',
    'OverallQualCategory'
]

In [ ]:
new_ordinal_numerical_features = [
    'Exter_Qual',
    'Kitchen_Qual',
    'OverallQualCategory'
]

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
X[['Overall_Qual', 'Quality_Score', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].dtypes

**Change data types of newly created `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [ ]:
for data in [X_train, X_test]:
    for col in new_categorical_features:
        data[col] = data[col].astype('category')

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
print(X[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].head())

In [ ]:
print(X[['Overall_Qual', 'OverallQualCategory', 'Exter_Qual', 'Kitchen_Qual']].dtypes)

In [ ]:
len(X.columns.to_list())

In [ ]:
X.head()

In [ ]:
new_categorical_features

In [ ]:
X[new_categorical_features].dtypes

In [ ]:
len(X.columns.to_list())

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

## 8. Mathematical Transformation

In [ ]:
skewed_features = X_train[continuous_num_features].skew() # .skew() is
skewed_features

In [ ]:
skewed_features = skewed_features[
    abs(skewed_features) > 0.75
]

In [ ]:
print(skewed_features)

In [ ]:
from sklearn.preprocessing import PowerTransformer

power_transformer = PowerTransformer(method='yeo-johnson')

power_transformer.fit(X_train[skewed_features.index])

X_train[skewed_features.index] = power_transformer.transform(
    X_train[skewed_features.index]
)


In [ ]:
X_test[skewed_features.index] = power_transformer.transform(
    X_test[skewed_features.index]
)

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

## 9. Categorical Encoding

In [ ]:
categorical_features = categorical_features + new_categorical_features

In [ ]:
len(categorical_features)

In [ ]:
categorical_features_final = categorical_features

In [ ]:
for col in categorical_features_final:
  print(f"{col}: {X[col].nunique()}")

In [ ]:
binary_columns = [
    col for col in categorical_features_final
    if X_train[col].nunique() == 2
]
binary_columns = list(set(binary_columns))
binary_columns

In [ ]:
for col in binary_columns:
    categories = X_train[col].dropna().unique()

    mapping = {categories[0]: 0, categories[1]: 1}

    X_train[col] = X_train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
ohe_cols = [
    col for col in categorical_features_final
    if X_train[col].nunique() > 2
]

In [ ]:
ohe_cols = list(set(ohe_cols))
ohe_cols

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_ohe = ohe.fit_transform(X_train[ohe_cols])
X_test_ohe = ohe.transform(X_test[ohe_cols])

In [ ]:
X_train_ohe = pd.DataFrame(
    X_train_ohe,
    columns=ohe.get_feature_names_out(ohe_cols),
    index=X_train.index
)

X_test_ohe = pd.DataFrame(
    X_test_ohe,
    columns=ohe.get_feature_names_out(ohe_cols),
    index=X_test.index
)

In [ ]:
X_train = pd.concat(
    [X_train.drop(columns=ohe_cols), X_train_ohe],
    axis=1
)

X_test = pd.concat(
    [X_test.drop(columns=ohe_cols), X_test_ohe],
    axis=1
)

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
len(X.columns.to_list())

> From $94$ to $369$ columns after encoding

## 10. Scaling

After encoding, most remaining features are numerical. Exclude the target

In [ ]:
numerical_features_all = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
len(numerical_features_all)

In [ ]:
features_to_exclude = set(temporal_features + X.select_dtypes(include=['object', 'category', 'int32']).columns.to_list())
features_to_exclude

In [ ]:
scaling_features = list(set(numerical_features_all) - set(features_to_exclude))

In [ ]:
len(scaling_features)

In [ ]:
X[scaling_features].info()

In [ ]:
X[scaling_features].head()
#

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

In [ ]:
minmax_scaler.fit(X_train[scaling_features])

In [ ]:
X_train[scaling_features] = minmax_scaler.transform(
    X_train[scaling_features]
)

In [ ]:
X_test[scaling_features] = minmax_scaler.transform(
    X_test[scaling_features]
)

In [ ]:
X = pd.concat([X_train, X_test], axis=0)

In [ ]:
X[scaling_features].head()

In [ ]:
X.head()

## 11. `Pipeline` and `ColumnTransformer`

While a `ColumnTransformer` is typically used earlier in the process to apply different transformations to different subsets of columns (e.g., numerical vs. categorical), our data (`df`) has already undergone comprehensive preprocessing, including handling missing values, scaling numerical features, and one-hot encoding categorical features.

At this stage, all features in our `df` are numerical and scaled, making a `ColumnTransformer` redundant for *further* preprocessing steps. However, a `Pipeline` is still invaluable for chaining model steps and ensuring consistency.

I will now re-split our preprocessed `df` into `X` (features) and `y` (target variable `SalePrice`), then define a simple `Pipeline` that includes a model, ready for training.

> On an other notebook

## 12. Modeling (Training)

### Models
**Baseline and Linear Models**
- Linear Regression (OLS)
- Ridge and Lasso Regression

**Tree-Based and Ensemble Models**
- Random Forest Regressor
- XGBoost (Extreme Gradient Boosting)
- LightGBM
- CatBoost

**Non-Linear Models**
- Support Vector Regressor (SVR)
### Evaluation Metrics
- **MAE (Mean Absolute Error)**: Measures average absolute distance between predictions and true values in original currency units.
- **RMSE (Root Mean Squared Error)**: Penalizes larger errors more heavily by squaring differences before taking the square root.
- **R² Score**: Represents the proportion of variance in house prices explained by the model features

In [ ]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
y_train.head()

In [ ]:
y_test.head()

**Import models**

In [ ]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
def evaluate_metrics(true, predicted):
    mse = mean_squared_error(true, predicted) # Calculate Accuracy
    mae = mean_absolute_error(true, predicted) # Calculated MAE
    rmse = np.sqrt(mse) # Calculated RMSE
    r2 = r2_score(true, predicted) # Calculated R2 Score
    return mse, mae, rmse, r2

In [ ]:
# Create a function which can evaluate models and return a report
def evaluate_regressor_models(X_train, y_train, X_test, y_test, models):
    '''
    This function takes in X_train, y_train, X_test, y_test and models dictionary as input
    Iterates through the given model dictionary and evaluates the metrics
    Returns: Dataframe which contains report of all models metrics with cost
    '''
    models_list = []
    mse_list = []

    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) # Train model

        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Training set performance
        mse_train, mae_train, rmse_train, r2_train = evaluate_metrics(y_train ,y_train_pred)


        # Test set performance
        mse_test, mae_test, rmse_test, r2_test = evaluate_metrics(y_test, y_test_pred)

        print(list(models.keys())[i])
        models_list.append(list(models.keys())[i])

        print('Model performance for Training set')
        print(f"- MSE: {mse_train:.4f}")
        print(f'- MAE: {mae_train:.4f}')
        print(f'- RMSE: {rmse_train:.4f}')
        print(f'- R2: {r2_train:.4f}')

        print('----------------------------------')

        print('Model performance for Test set')
        print(f'- MSE: {mse_test:.4f}')
        mse_list.append(mse_test)
        print(f'- MAE: {mae_test:.4f}')
        print(f'- RMSE: {rmse_test:.4f}')
        print(f'- R2: {r2_test:.4f}')
        print('='*35)
        print('\n')

    report=pd.DataFrame(list(zip(models_list, mse_list)), columns=['Model Name', 'MSE']).sort_values(by=['MSE'], ascending=True) # Sort by MSE ascending, lower is better

    return report

In [ ]:
import pandas as pd

for col in X_train.columns:
    if pd.api.types.is_categorical_dtype(X_train[col]):
        # Check if the categories themselves are numerical or boolean
        # This handles cases like 'Street' (after mapping to 0/1) or 'HasPool'
        if (pd.api.types.is_numeric_dtype(X_train[col].cat.categories) or
            pd.api.types.is_bool_dtype(X_train[col].cat.categories)):
            X_train[col] = X_train[col].astype(int)
            X_test[col] = X_test[col].astype(int)

X = pd.concat([X_train, X_test], axis=0)

In [ ]:

base_model_report = evaluate_regressor_models(X_train, y_train, X_test, y_test, models)

In [ ]:
base_model_report

**Model Performance Visualization**

In [ ]:
print('\nModel Performance Visualization')
for model_name, regressor in models.items():
    # Ensure the model is trained before predicting
    # The evaluate_regressor_models function already trains the models
    # So, we can directly predict using the models in the 'models' dictionary.
    y_pred = regressor.predict(X_test)

    plt.figure(figsize=(16, 6))

    # Regression Plot
    plt.subplot(1, 2, 1)
    sns.regplot(x=y_pred, y=y_test, scatter_kws={'alpha':0.4})
    plt.xlabel('Predicted Price')
    plt.ylabel('Actual Price')
    plt.title(f'Regression Plot: Actual vs Predicted ({model_name})')

    # Residual Plot
    plt.subplot(1, 2, 2)
    residuals = y_test - y_pred
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.4)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted Price')
    plt.ylabel('Residuals')
    plt.title(f'Residual Plot ({model_name})')

    plt.tight_layout()
    plt.show()

**Cross Validation**

In [ ]:
from sklearn.model_selection import cross_validate

print("Cross-Validation Results:")
for model_name, regressor in models.items():
    scoring = {'mae': 'neg_mean_absolute_error', 'mse': 'neg_mean_squared_error', 'r2': 'r2'}
    cv_results = cross_validate(regressor, X, y, cv=5, scoring=scoring)
    print(f"\n{model_name}:")
    print(f"  Average MSE: {-cv_results['test_mse'].mean():.4f}")
    print(f"  Average MAE: {-cv_results['test_mae'].mean():.4f}")
    print(f"  Average RMSE: {np.sqrt(-cv_results['test_mse'].mean()):.4f}")
    print(f"  Average R2: {cv_results['test_r2'].mean():.4f}")

**Hyperparameter Tuning**

In [ ]:
# default parameters
print("Default Parameters:")
for model_name, regressor in models.items():
    print(f'{model_name}')
    print(f'parameters: {regressor.get_params()}', end='\n\n')

In [ ]:
# Define parameter grids for hyperparameter tuning
model_params = [
    ('Linear Regression', LinearRegression(), {
        'fit_intercept': [True, False]
    }),
    ('Lasso Regression', Lasso(), {
        'alpha': [0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    }),
    ('Ridge Regression', Ridge(), {
        'alpha': [0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    }),
    ('Random Forest', RandomForestRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    }),
    ('GBM', GradientBoostingRegressor(random_state=42), {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'max_depth': [3, 5, 7]
    }),
    ('XGBoost', XGBRegressor(random_state=42), {
        'n_estimators': [200, 400],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 1.0]
    })
]

In [ ]:
# Best parameters and scores initialization
best_model_name = None
best_model = None
best_score = float('-inf')
best_params = None

print("\nHyperparameter Tuning Results:")
for model_name, regressor, params in model_params:
    regressor_grid = GridSearchCV(regressor, params, cv=5, n_jobs=-1, verbose=1).fit(X, y)

    print(f"\nModel: {model_name}")
    print("Best Parameters:", regressor_grid.best_params_)
    print("Best Score:", regressor_grid.best_score_)
    print("\n")

    if regressor_grid.best_score_ > best_score:
        best_model_name = model_name
        best_model = regressor
        best_score = regressor_grid.best_score_
        best_params = regressor_grid.best_params_

print("\n\nFinal Inference")
print(f"Model: {best_model_name}")
print("Best Parameters:", best_params)
print("Best Score:", best_score)


In [ ]:
best_model

In [ ]:
# Function to find best model parameters
def best_params(model_params, best_model):
    for model_name, regressor, params in model_params:
        if best_model == regressor:
            return params
    return None

In [ ]:
# Apply the function
best_model_params = best_params(model_params, best_model)

In [ ]:
best_model_params

In [ ]:
# Final GridSearchCV with best model and its parameter grid
regressor_best_grid = GridSearchCV(best_model, best_model_params, cv=5, n_jobs=-1, verbose=1).fit(X, y)

In [ ]:
regressor_best_grid

In [ ]:
final_model = GradientBoostingRegressor(
    **regressor_best_grid.best_params_
)

final_model.fit(X_train, y_train)

In [ ]:
y_pred = final_model.predict(X_test)

In [ ]:
y_pred

**Cross Validation**

In [ ]:
print("Best Model Cross-Validation")

scoring = {
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error',
    'r2': 'r2'
}

cv_results_final = cross_validate(
    final_model,
    X_train,
    y_train,
    cv=5,
    scoring=scoring
)

In [ ]:
print("\nGradient Boosting:")

print(
    f"  Average MAE: "
    f"{-cv_results_final['test_mae'].mean():.4f}"
)

print(
    f"  Average MSE: "
    f"{-cv_results_final['test_mse'].mean():.4f}"
)

print(
    f"  Average RMSE: "
    f"{np.sqrt(-cv_results_final['test_mse'].mean()):.4f}"
)

print(
    f"  Average R2: "
    f"{cv_results_final['test_r2'].mean():.4f}"
)

**Feature evaluation on `X_test`**

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_test_pred = final_model.predict(X_test)

mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_test_pred)

print("Final Test Performance")
print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

**Feature Importance**

In [ ]:
def visualize_importance(model, features, start=0, num=20):

    feature_imp = pd.DataFrame({
        'Feature': features.columns,
        'Importance': model.feature_importances_
    })

    feature_imp = feature_imp.sort_values(
        by='Importance',
        ascending=False
    )

    selected = feature_imp.iloc[start:num]

    display(selected)

    plt.figure(figsize=(12, 8))

    sns.barplot(
        x='Importance',
        y='Feature',
        data=selected
    )

    plt.title('Feature Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_importance(
    final_model,
    X_train,
    start=0,
    num=45
)

In [ ]:
predictions = final_model.predict(X_test)

In [ ]:
predictions

In [ ]:
test_sample = X_test.iloc[0].values.reshape(1, -1)
prediction = final_model.predict(test_sample)
print(prediction)

In [ ]:
predict_df = X_test.copy()

##13. Saving model with Pickle

In [ ]:
import pickle

with open('ames_ml_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

print("Model saved to ames_ml_model.pkl")

**Loading model and predicting using it**

In [ ]:
# Load model from pickle file
with open('ames_ml_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

result = loaded_model.score(X_test, y_test)
print(result)

print('Predicted SalePrice is : ', loaded_model.predict(predict_df))